# HDBSCAN Extra Analysis

This notebook is for exploratory analysis on top of the HDBSCAN asset impact pipeline outputs.

It does not rebuild the pipeline tables. It assumes the HDBSCAN pipeline notebook has already been run and that the DuckDB tables exist.

In [23]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter, PercentFormatter

from IPython.display import Markdown, display

DB_PATH = 'developer_project.duckdb'
con = duckdb.connect(DB_PATH)

CLUSTER_PROFILE_TABLE = 'cluster_profile_asset_base_v2'
PROFILE_TABLE = CLUSTER_PROFILE_TABLE
CLUSTER_ASSET_LONG_TABLE = 'cluster_asset_long_v2'
CLUSTER_ASSET_SUMMARY_TABLE = 'cluster_asset_summary_v2'
CLUSTER_ASSET_PRIORITY_TABLE = 'cluster_asset_priority_by_cluster_v2'
CLUSTER_PERSONA_PROFILE_TABLE = 'cluster_persona_profile_v2'
CLUSTER_JOURNEY_PROFILE_TABLE = 'cluster_journey_profile_v2'
CLUSTER_EFFORT_PROFILE_TABLE = 'cluster_effort_profile_v2'
CLUSTER_TOP_PERSONA_SUMMARY_TABLE = 'cluster_top_persona_summary_v2'
CLUSTER_TOP_ASSET_SUMMARY_TABLE = 'cluster_top_asset_summary_v2'
CLUSTER_CORRELATION_SUMMARY_TABLE = 'cluster_correlation_summary_v2'
CLUSTER_GROUP_ROLLUP_TABLE = 'cluster_group_rollup_summary_v2'
CLUSTER_GROUP_ASSET_PROFILE_TABLE = 'cluster_group_asset_profile_v2'
CLUSTER_GROUP_COMPOSITION_TABLE = 'cluster_group_composition_summary_v2'
ASSET_AUDIENCE_TABLE = 'asset_audience_summary_v2'


In [24]:
required_tables = [
    CLUSTER_PROFILE_TABLE,
    CLUSTER_ASSET_LONG_TABLE,
    CLUSTER_ASSET_SUMMARY_TABLE,
    CLUSTER_ASSET_PRIORITY_TABLE,
    CLUSTER_PERSONA_PROFILE_TABLE,
    CLUSTER_JOURNEY_PROFILE_TABLE,
    CLUSTER_EFFORT_PROFILE_TABLE,
    CLUSTER_TOP_PERSONA_SUMMARY_TABLE,
    CLUSTER_TOP_ASSET_SUMMARY_TABLE,
    CLUSTER_CORRELATION_SUMMARY_TABLE,
    CLUSTER_GROUP_ROLLUP_TABLE,
    CLUSTER_GROUP_ASSET_PROFILE_TABLE,
    CLUSTER_GROUP_COMPOSITION_TABLE,
    ASSET_AUDIENCE_TABLE,
]

missing = []
for table_name in required_tables:
    try:
        con.execute(f'SELECT 1 FROM {table_name} LIMIT 1')
    except Exception:
        missing.append(table_name)

if missing:
    raise ValueError(f'Missing pipeline output tables: {missing}')

display(Markdown('### Pipeline Tables Available'))
display(pd.DataFrame({'table_name': required_tables}))

### Pipeline Tables Available

,table_name
0,cluster_profile_asset_base_v2
1,cluster_asset_priority_by_cluster_v2
2,cluster_persona_profile_v2
3,cluster_journey_profile_v2
4,cluster_effort_profile_v2
5,cluster_top_persona_summary_v2
6,cluster_top_asset_summary_v2
7,cluster_correlation_summary_v2
8,cluster_group_rollup_summary_v2
9,cluster_group_asset_profile_v2


## Lifecycle Group Analysis

These views keep the analysis at the `active`, `cooling`, and `at_risk` level.

In [25]:
display(Markdown('### Lifecycle Group Rollup Summary'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_GROUP_ROLLUP_TABLE}
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
""").fetchdf())

display(Markdown('### Lifecycle Group Asset Profile'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_GROUP_ASSET_PROFILE_TABLE}
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
""").fetchdf())

display(Markdown('### Lifecycle Group Composition Table'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_GROUP_COMPOSITION_TABLE}
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
""").fetchdf())

### Lifecycle Group Rollup Summary

,cluster_group,developers,top_persona_1,top_persona_1_share,top_persona_2,top_persona_2_share,top_persona_3,top_persona_3_share,top_effort_1,top_effort_1_share,top_effort_2,top_effort_2_share,top_volume_asset_1,top_volume_asset_2,top_volume_asset_3,top_breadth_asset_1,top_intensity_asset_1
0,active,418049,GenAI,0.698562,CUDA,0.138912,Robotics,0.060670,very high effort,0.962586,low effort,0.031159,ngc_download,devzone_download,forum_contribution,devzone_download,ngc_download
1,cooling,356500,GenAI,0.434056,CUDA,0.285316,Robotics,0.094003,very high effort,0.535352,high effort,0.390418,devzone_download,ngc_download,dli_training,devzone_download,ngc_download
2,at_risk,1580877,CUDA,0.408582,GenAI,0.340516,Robotics,0.082461,low effort,0.387062,medium effort,0.307890,devzone_download,ngc_download,dli_training,devzone_download,ngc_download


### Lifecycle Group Asset Profile

,cluster_group,developers,pct_dli_training,lift_dli_training,intensity_dli_training,pct_webinar,lift_webinar,intensity_webinar,pct_forum_contribution,lift_forum_contribution,intensity_forum_contribution,pct_bug_filed,lift_bug_filed,intensity_bug_filed,pct_hackathon,lift_hackathon,intensity_hackathon,pct_devzone_download,lift_devzone_download,intensity_devzone_download,pct_ngc_download,lift_ngc_download,intensity_ngc_download
0,active,418049,0.086629,0.733580,2.564904,0.016890,0.822280,2.366520,0.010542,1.264678,69.153846,0.001526,1.746666,30.040752,0.000033,0.268460,1.071429,0.174097,0.412078,40.791566,0.017670,1.314477,732.196426
1,cooling,356500,0.183868,1.557015,2.034570,0.042275,2.058084,1.847787,0.013966,1.675509,12.721028,0.001433,1.640506,19.906067,0.000070,0.562160,1.280000,0.339518,0.803620,25.668286,0.018463,1.373443,74.474020
2,at_risk,1580877,0.243432,2.061408,1.700735,0.037057,1.804042,1.609914,0.010389,1.246288,8.032698,0.001567,1.793988,20.974576,0.000083,0.669353,1.098485,0.409577,0.969447,14.468510,0.023156,1.722529,26.366361


### Lifecycle Group Composition Table

,cluster_group,developers,top_persona_1,top_persona_1_share,top_persona_2,top_persona_2_share,top_persona_3,top_persona_3_share,top_journey_1,top_journey_1_share,top_journey_2,top_journey_2_share,top_effort_1,top_effort_1_share,top_effort_2,top_effort_2_share
0,active,418049,GenAI,0.698562,CUDA,0.138912,Robotics,0.060670,Evaluator,0.671924,Learner,0.204490,very high effort,0.962586,low effort,0.031159
1,cooling,356500,GenAI,0.434056,CUDA,0.285316,Robotics,0.094003,Historically_Active,0.929719,Builder,0.068519,very high effort,0.535352,high effort,0.390418
2,at_risk,1580877,CUDA,0.408582,GenAI,0.340516,Robotics,0.082461,Historically_Active,0.948792,Builder,0.048848,low effort,0.387062,medium effort,0.307890


## Cluster Analysis

These views stay at the per-cluster level so the lifecycle group summaries can be tied back to concrete cluster patterns.

In [26]:
display(Markdown('### Top Personas By Cluster'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_TOP_PERSONA_SUMMARY_TABLE}
ORDER BY cluster_label
""").fetchdf())

display(Markdown('### Top Assets By Cluster'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_TOP_ASSET_SUMMARY_TABLE}
ORDER BY cluster_label
""").fetchdf())

display(Markdown('### Cluster Correlation Summary'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_CORRELATION_SUMMARY_TABLE}
ORDER BY cluster_group, cluster_developers DESC, cluster_label
""").fetchdf())

### Top Personas By Cluster

,cluster_label,top_persona_1,top_persona_1_share,top_persona_2,top_persona_2_share,top_persona_3,top_persona_3_share
0,cluster_Dormant_Former_Builders,CUDA,0.686126,Simulation,0.105529,Unknown,0.075067
1,cluster_Dormant_Low_Depth,CUDA,0.529681,GenAI,0.193532,Learning_Community,0.098306
2,cluster_Dormant_One_Time_Users,GenAI,0.371202,CUDA,0.230721,Unknown,0.192822
3,cluster_active_0,GenAI,0.845394,Unknown,0.104499,Robotics,0.019119
4,cluster_active_1,GenAI,0.957440,CUDA,0.018236,Unknown,0.016511
5,cluster_active_2,GenAI,0.975234,Unknown,0.022759,CUDA,0.000816
6,cluster_active_3,GenAI,0.460634,Learning_Community,0.197055,CUDA,0.167647
7,cluster_active_4,GenAI,0.832381,Robotics,0.081330,CUDA,0.059879
8,cluster_active_5,CUDA,0.583902,GenAI,0.153317,Unknown,0.115348
9,cluster_active_noise,CUDA,0.376276,GenAI,0.367952,Robotics,0.162037


### Top Assets By Cluster

,cluster_label,top_volume_asset_1,top_volume_asset_2,top_volume_asset_3,top_breadth_asset_1,top_breadth_asset_2,top_breadth_asset_3,top_intensity_asset_1,top_intensity_asset_2,top_intensity_asset_3
0,cluster_Dormant_Former_Builders,devzone_download,ngc_download,forum_contribution,devzone_download,ngc_download,dli_training,ngc_download,devzone_download,forum_contribution
1,cluster_Dormant_Low_Depth,devzone_download,dli_training,forum_contribution,devzone_download,dli_training,webinar,bug_filed,forum_contribution,ngc_download
2,cluster_Dormant_One_Time_Users,webinar,devzone_download,dli_training,webinar,devzone_download,dli_training,webinar,devzone_download,dli_training
3,cluster_active_0,webinar,dli_training,devzone_download,webinar,dli_training,devzone_download,bug_filed,devzone_download,dli_training
4,cluster_active_1,ngc_download,webinar,devzone_download,ngc_download,webinar,devzone_download,ngc_download,webinar,devzone_download
5,cluster_active_2,forum_contribution,ngc_download,devzone_download,forum_contribution,ngc_download,devzone_download,forum_contribution,ngc_download,devzone_download
6,cluster_active_3,dli_training,devzone_download,forum_contribution,dli_training,devzone_download,forum_contribution,bug_filed,forum_contribution,devzone_download
7,cluster_active_4,devzone_download,webinar,forum_contribution,devzone_download,webinar,forum_contribution,devzone_download,webinar,forum_contribution
8,cluster_active_5,devzone_download,ngc_download,webinar,devzone_download,ngc_download,webinar,ngc_download,forum_contribution,devzone_download
9,cluster_active_noise,ngc_download,devzone_download,forum_contribution,devzone_download,dli_training,ngc_download,ngc_download,forum_contribution,devzone_download


### Cluster Correlation Summary

,cluster_label,cluster_group,cluster_developers,top_persona_1,top_persona_1_share,top_persona_2,top_persona_2_share,top_persona_3,top_persona_3_share,dominant_effort,dominant_effort_share,dominant_journey,dominant_journey_share,top_volume_asset_1,top_volume_asset_2,top_volume_asset_3,top_breadth_asset_1,top_breadth_asset_2,top_breadth_asset_3,top_intensity_asset_1,top_intensity_asset_2,top_intensity_asset_3,dominant_segment_persona,dominant_segment_effort,dominant_segment_journey,dominant_segment_developers,dominant_segment_share,dominant_segment_top_volume_asset,dominant_segment_top_breadth_asset,dominant_segment_top_intensity_asset
0,cluster_active_0,active,155026,GenAI,0.845394,Unknown,0.104499,Robotics,0.019119,very high effort,1.000000,Evaluator,0.984951,webinar,dli_training,devzone_download,webinar,dli_training,devzone_download,bug_filed,devzone_download,dli_training,GenAI,very high effort,Evaluator,130355,1.0,webinar,webinar,bug_filed
1,cluster_active_noise,active,105723,CUDA,0.376276,GenAI,0.367952,Robotics,0.162037,very high effort,0.852132,Learner,0.508366,ngc_download,devzone_download,forum_contribution,devzone_download,dli_training,ngc_download,ngc_download,forum_contribution,devzone_download,GenAI,very high effort,Learner,19558,1.0,dli_training,dli_training,devzone_download
2,cluster_active_1,active,66683,GenAI,0.957440,CUDA,0.018236,Unknown,0.016511,very high effort,0.999880,Evaluator,0.999460,ngc_download,webinar,devzone_download,ngc_download,webinar,devzone_download,ngc_download,webinar,devzone_download,GenAI,very high effort,Evaluator,63829,1.0,ngc_download,ngc_download,ngc_download
3,cluster_active_2,active,29395,GenAI,0.975234,Unknown,0.022759,CUDA,0.000816,very high effort,1.000000,Learner,0.999694,forum_contribution,ngc_download,devzone_download,forum_contribution,ngc_download,devzone_download,forum_contribution,ngc_download,devzone_download,GenAI,very high effort,Learner,28667,1.0,forum_contribution,forum_contribution,forum_contribution
4,cluster_active_3,active,24653,GenAI,0.460634,Learning_Community,0.197055,CUDA,0.167647,very high effort,1.000000,Evaluator,0.997769,dli_training,devzone_download,forum_contribution,dli_training,devzone_download,forum_contribution,bug_filed,forum_contribution,devzone_download,GenAI,very high effort,Evaluator,11347,1.0,dli_training,dli_training,bug_filed
5,cluster_active_4,active,18554,GenAI,0.832381,Robotics,0.081330,CUDA,0.059879,very high effort,1.000000,Evaluator,0.998814,devzone_download,webinar,forum_contribution,devzone_download,webinar,forum_contribution,devzone_download,webinar,forum_contribution,Robotics,very high effort,Evaluator,1507,1.0,forum_contribution,forum_contribution,forum_contribution
6,cluster_active_5,active,18015,CUDA,0.583902,GenAI,0.153317,Unknown,0.115348,very high effort,1.000000,Builder,1.000000,devzone_download,ngc_download,webinar,devzone_download,ngc_download,webinar,ngc_download,forum_contribution,devzone_download,CUDA,very high effort,Builder,10519,1.0,devzone_download,devzone_download,ngc_download
7,cluster_at_risk_0,at_risk,436295,CUDA,0.387119,GenAI,0.342560,Robotics,0.096318,high effort,0.437016,Historically_Active,0.944987,devzone_download,dli_training,ngc_download,devzone_download,dli_training,webinar,ngc_download,devzone_download,forum_contribution,CUDA,high effort,Historically_Active,84485,1.0,devzone_download,devzone_download,forum_contribution
8,cluster_at_risk_1,at_risk,365945,CUDA,0.552203,GenAI,0.201175,Robotics,0.131659,medium effort,0.523398,Historically_Active,0.897307,devzone_download,dli_training,ngc_download,devzone_download,dli_training,webinar,devzone_download,ngc_download,forum_contribution,CUDA,medium effort,Historically_Active,113169,1.0,devzone_download,devzone_download,devzone_download
9,cluster_at_risk_2,at_risk,210377,GenAI,0.584517,CUDA,0.168673,Learning_Community,0.102725,medium effort,0.760164,Historically_Active,1.000000,dli_training,devzone_download,webinar,dli_training,devzone_download,webinar,devzone_down

In [27]:
display(Markdown('### Cluster Comparison View'))
display(con.execute(f"""
SELECT
    cluster_label,
    cluster_group,
    cluster_developers,
    top_persona_1,
    top_persona_1_share,
    dominant_journey,
    dominant_journey_share,
    dominant_effort,
    dominant_effort_share,
    top_volume_asset_1,
    top_volume_asset_2,
    top_breadth_asset_1,
    top_intensity_asset_1
FROM {CLUSTER_CORRELATION_SUMMARY_TABLE}
ORDER BY cluster_group, cluster_developers DESC, cluster_label
""").fetchdf())

### Cluster Comparison View

,cluster_label,cluster_group,cluster_developers,top_persona_1,top_persona_1_share,dominant_journey,dominant_journey_share,dominant_effort,dominant_effort_share,top_volume_asset_1,top_volume_asset_2,top_breadth_asset_1,top_intensity_asset_1
0,cluster_active_0,active,155026,GenAI,0.845394,Evaluator,0.984951,very high effort,1.000000,webinar,dli_training,webinar,bug_filed
1,cluster_active_noise,active,105723,CUDA,0.376276,Learner,0.508366,very high effort,0.852132,ngc_download,devzone_download,devzone_download,ngc_download
2,cluster_active_1,active,66683,GenAI,0.957440,Evaluator,0.999460,very high effort,0.999880,ngc_download,webinar,ngc_download,ngc_download
3,cluster_active_2,active,29395,GenAI,0.975234,Learner,0.999694,very high effort,1.000000,forum_contribution,ngc_download,forum_contribution,forum_contribution
4,cluster_active_3,active,24653,GenAI,0.460634,Evaluator,0.997769,very high effort,1.000000,dli_training,devzone_download,dli_training,bug_filed
5,cluster_active_4,active,18554,GenAI,0.832381,Evaluator,0.998814,very high effort,1.000000,devzone_download,webinar,devzone_download,devzone_download
6,cluster_active_5,active,18015,CUDA,0.583902,Builder,1.000000,very high effort,1.000000,devzone_download,ngc_download,devzone_download,ngc_download
7,cluster_at_risk_0,at_risk,436295,CUDA,0.387119,Historically_Active,0.944987,high effort,0.437016,devzone_download,dli_training,devzone_download,ngc_download
8,cluster_at_risk_1,at_risk,365945,CUDA,0.552203,Historically_Active,0.897307,medium effort,0.523398,devzone_download,dli_training,devzone_download,devzone_download
9,cluster_at_risk_2,at_risk,210377,GenAI,0.584517,Historically_Active,1.000000,medium effort,0.760164,dli_training,devzone_download,dli_training,devzone_download


## Asset-Centered Analysis

This flips the direction of the analysis from `group -> assets` to `asset -> audience`.

In [28]:
display(Markdown('### Asset Audience Table'))
display(con.execute(f"""
SELECT *
FROM {ASSET_AUDIENCE_TABLE}
ORDER BY exposed_developers DESC, asset_name
""").fetchdf())

display(Markdown('### Asset Audience: Sorted By Active Share'))
display(con.execute(f"""
SELECT *
FROM {ASSET_AUDIENCE_TABLE}
ORDER BY pct_active DESC, exposed_developers DESC, asset_name
""").fetchdf())

display(Markdown('### Asset Audience: Sorted By Learner Share'))
display(con.execute(f"""
SELECT *
FROM {ASSET_AUDIENCE_TABLE}
ORDER BY pct_learner DESC, exposed_developers DESC, asset_name
""").fetchdf())

### Asset Audience Table

,asset_name,asset_group,asset_role,exposed_developers,pct_active,pct_cooling,pct_at_risk,pct_builder,pct_learner,pct_high_effort,top_persona
0,devzone_download,activity_asset,download_activity,3962571,0.018367,0.030545,0.163402,0.077534,0.004258,0.248460,CUDA
1,dli_training,activity_asset,learning_activity,1107590,0.032697,0.059182,0.347453,0.029501,0.007366,0.404712,GenAI
2,webinar,activity_asset,learning_activity,192657,0.036651,0.078227,0.304074,0.072004,0.016449,0.441816,GenAI
3,ngc_download,activity_asset,download_activity,126082,0.058589,0.052204,0.290335,0.317857,0.006163,0.574539,GenAI
4,forum_contribution,activity_asset,community_activity,78181,0.056369,0.063686,0.210064,0.329978,0.014991,0.658894,CUDA
5,bug_filed,activity_asset,community_activity,8195,0.077852,0.062355,0.302379,0.259671,0.006711,0.738133,CUDA
6,hackathon,activity_asset,community_activity,1170,0.011966,0.021368,0.112821,0.128205,0.002564,0.271795,CUDA


### Asset Audience: Sorted By Active Share

,asset_name,asset_group,asset_role,exposed_developers,pct_active,pct_cooling,pct_at_risk,pct_builder,pct_learner,pct_high_effort,top_persona
0,bug_filed,activity_asset,community_activity,8195,0.077852,0.062355,0.302379,0.259671,0.006711,0.738133,CUDA
1,ngc_download,activity_asset,download_activity,126082,0.058589,0.052204,0.290335,0.317857,0.006163,0.574539,GenAI
2,forum_contribution,activity_asset,community_activity,78181,0.056369,0.063686,0.210064,0.329978,0.014991,0.658894,CUDA
3,webinar,activity_asset,learning_activity,192657,0.036651,0.078227,0.304074,0.072004,0.016449,0.441816,GenAI
4,dli_training,activity_asset,learning_activity,1107590,0.032697,0.059182,0.347453,0.029501,0.007366,0.404712,GenAI
5,devzone_download,activity_asset,download_activity,3962571,0.018367,0.030545,0.163402,0.077534,0.004258,0.248460,CUDA
6,hackathon,activity_asset,community_activity,1170,0.011966,0.021368,0.112821,0.128205,0.002564,0.271795,CUDA


### Asset Audience: Sorted By Learner Share

,asset_name,asset_group,asset_role,exposed_developers,pct_active,pct_cooling,pct_at_risk,pct_builder,pct_learner,pct_high_effort,top_persona
0,webinar,activity_asset,learning_activity,192657,0.036651,0.078227,0.304074,0.072004,0.016449,0.441816,GenAI
1,forum_contribution,activity_asset,community_activity,78181,0.056369,0.063686,0.210064,0.329978,0.014991,0.658894,CUDA
2,dli_training,activity_asset,learning_activity,1107590,0.032697,0.059182,0.347453,0.029501,0.007366,0.404712,GenAI
3,bug_filed,activity_asset,community_activity,8195,0.077852,0.062355,0.302379,0.259671,0.006711,0.738133,CUDA
4,ngc_download,activity_asset,download_activity,126082,0.058589,0.052204,0.290335,0.317857,0.006163,0.574539,GenAI
5,devzone_download,activity_asset,download_activity,3962571,0.018367,0.030545,0.163402,0.077534,0.004258,0.248460,CUDA
6,hackathon,activity_asset,community_activity,1170,0.011966,0.021368,0.112821,0.128205,0.002564,0.271795,CUDA


## Notes

- Keep adding exploratory tables here first.
- Once a table proves useful, decide later whether it belongs back in the pipeline.
- This notebook is the safer place for long-form interpretation and recommendation writing.

## Asset Impact by Lifecycle Group

This section applies the exposed-vs-unexposed and pre/post delta logic from `AssetImpact_Analysis_Runnable.ipynb` but groups by HDBSCAN lifecycle group (`active`, `cooling`, `at_risk`).

**What this answers:**
- Within each lifecycle group, do asset-exposed developers show stronger recent behavior than unexposed developers with the same baseline?
- Which asset exposures are associated with higher adoption outcomes (API usage, activity velocity)?
- Which asset combinations co-occur in each group?

**Caveats:**
- This is observational, not causal. Exposure and outcomes are measured from the same cross-sectional snapshot.
- Time windows (30-90d as baseline, 0-30d as recent) are a proxy for before/after, not true sequencing.
- `dev_profile_final_v4` has lifetime and windowed counts but no event-level timestamps.

### 0b. Raw Asset Exposure Rates by Cluster

What percentage of developers in each cluster have **ever** used each asset type? No normalization — just binary "touched it or not" rates. This answers the question the volume/breadth/intensity metrics obscure: how many developers in each cluster actually used each asset?

In [29]:
# ── Raw asset exposure rates: % of developers who ever used each asset ──

cluster_group_case = """
CASE
    WHEN cluster_label LIKE 'cluster_active_%' THEN 'active'
    WHEN cluster_label LIKE 'cluster_cooling_%' THEN 'cooling'
    WHEN cluster_label LIKE 'cluster_at_risk_%' THEN 'at_risk'
    ELSE NULL
END
"""

# ── By lifecycle group ──
display(Markdown('### Raw Exposure Rates by Lifecycle Group'))

group_exposure_sql = f"""
SELECT
    {cluster_group_case} AS cluster_group,
    COUNT(*) AS developers,
    SUM(CASE WHEN lifetime_dli_training_count > 0 THEN 1 ELSE 0 END) AS n_training,
    ROUND(AVG(CASE WHEN lifetime_dli_training_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_training,
    SUM(CASE WHEN lifetime_webinar_count > 0 THEN 1 ELSE 0 END) AS n_webinar,
    ROUND(AVG(CASE WHEN lifetime_webinar_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_webinar,
    SUM(CASE WHEN lifetime_devzone_download_count > 0 THEN 1 ELSE 0 END) AS n_devzone_dl,
    ROUND(AVG(CASE WHEN lifetime_devzone_download_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_devzone_dl,
    SUM(CASE WHEN lifetime_ngc_download_count > 0 THEN 1 ELSE 0 END) AS n_ngc_dl,
    ROUND(AVG(CASE WHEN lifetime_ngc_download_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_ngc_dl,
    SUM(CASE WHEN lifetime_forum_count > 0 THEN 1 ELSE 0 END) AS n_forum,
    ROUND(AVG(CASE WHEN lifetime_forum_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_forum,
    SUM(CASE WHEN lifetime_bug_count > 0 THEN 1 ELSE 0 END) AS n_bug,
    ROUND(AVG(CASE WHEN lifetime_bug_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_bug,
    SUM(CASE WHEN lifetime_hackathon_count > 0 THEN 1 ELSE 0 END) AS n_hackathon,
    ROUND(AVG(CASE WHEN lifetime_hackathon_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_hackathon
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1
ORDER BY 1
"""

group_exposure = con.execute(group_exposure_sql).fetchdf()
display(group_exposure)

# ── By individual cluster ──
display(Markdown('### Raw Exposure Rates by Cluster'))

cluster_exposure_sql = f"""
SELECT
    cluster_label,
    {cluster_group_case} AS cluster_group,
    COUNT(*) AS developers,
    ROUND(AVG(CASE WHEN lifetime_dli_training_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_training,
    ROUND(AVG(CASE WHEN lifetime_webinar_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_webinar,
    ROUND(AVG(CASE WHEN lifetime_devzone_download_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_devzone_dl,
    ROUND(AVG(CASE WHEN lifetime_ngc_download_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_ngc_dl,
    ROUND(AVG(CASE WHEN lifetime_forum_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_forum,
    ROUND(AVG(CASE WHEN lifetime_bug_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_bug,
    ROUND(AVG(CASE WHEN lifetime_hackathon_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_hackathon,
    ROUND(AVG(CASE WHEN activity_count_0_30d > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_recent_activity,
    ROUND(AVG(CASE WHEN high_effort_count_0_30d > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_recent_high_effort
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1, 2
ORDER BY cluster_group, developers DESC
"""

cluster_exposure = con.execute(cluster_exposure_sql).fetchdf()
display(cluster_exposure)

# ── Heatmap-style view: just the percentages pivoted ──
display(Markdown('### Exposure Rate Heatmap (percentages only)'))

pct_cols = ['pct_training', 'pct_webinar', 'pct_devzone_dl', 'pct_ngc_dl', 'pct_forum', 'pct_bug', 'pct_hackathon']
heatmap = cluster_exposure.set_index('cluster_label')[pct_cols].copy()
heatmap.columns = ['Training', 'Webinar', 'DevZone DL', 'NGC DL', 'Forum', 'Bug Filed', 'Hackathon']

styled = heatmap.style.background_gradient(cmap='YlOrRd', axis=None).format('{:.1%}')
display(styled)

### Raw Exposure Rates by Lifecycle Group

,cluster_group,developers,n_training,pct_training,n_webinar,pct_webinar,n_devzone_dl,pct_devzone_dl,n_ngc_dl,pct_ngc_dl,n_forum,pct_forum,n_bug,pct_bug,n_hackathon,pct_hackathon
0,active,418049,36215.0,0.0866,7061.0,0.0169,72781.0,0.1741,7387.0,0.0177,4407.0,0.0105,638.0,0.0015,14.0,0.0000
1,at_risk,1580877,384836.0,0.2434,58582.0,0.0371,647491.0,0.4096,36606.0,0.0232,16423.0,0.0104,2478.0,0.0016,132.0,0.0001
2,cooling,356500,65549.0,0.1839,15071.0,0.0423,121038.0,0.3395,6582.0,0.0185,4979.0,0.0140,511.0,0.0014,25.0,0.0001


### Raw Exposure Rates by Cluster

,cluster_label,cluster_group,developers,pct_training,pct_webinar,pct_devzone_dl,pct_ngc_dl,pct_forum,pct_bug,pct_hackathon,pct_recent_activity,pct_recent_high_effort
0,cluster_active_0,active,155026,0.0069,0.0098,0.0029,0.0000,0.0001,0.0001,0.0000,1.0,0.9850
1,cluster_active_noise,active,105723,0.1537,0.0506,0.4650,0.0694,0.0381,0.0058,0.0001,1.0,0.2577
2,cluster_active_1,active,66683,0.0000,0.0001,0.0001,0.0002,0.0000,0.0000,0.0000,1.0,0.9995
3,cluster_active_2,active,29395,0.0000,0.0000,0.0000,0.0001,0.0006,0.0000,0.0000,1.0,0.0003
4,cluster_active_3,active,24653,0.7662,0.0067,0.2166,0.0003,0.0137,0.0000,0.0000,1.0,0.9978
5,cluster_active_4,active,18554,0.0000,0.0004,0.0006,0.0000,0.0001,0.0000,0.0000,1.0,0.9988
6,cluster_active_5,active,18015,0.0001,0.0008,0.9887,0.0012,0.0001,0.0000,0.0001,1.0,0.9968
7,cluster_at_risk_0,at_risk,436295,0.2544,0.0434,0.4173,0.0083,0.0112,0.0005,0.0001,0.0,0.0000
8,cluster_at_risk_1,at_risk,365945,0.2023,0.0456,0.6688,0.0203,0.0171,0.0007,0.0001,0.0,0.0000
9,cluster_at_risk_2,at_risk,210377,0.7178,0.0101,0.1444,0.0000,0.0044,0.0000,0.0000,0.0,0.0000


### Exposure Rate Heatmap (percentages only)

,Training,Webinar,DevZone DL,NGC DL,Forum,Bug Filed,Hackathon
cluster_label,,,,,,,
cluster_active_0,0.7%,1.0%,0.3%,0.0%,0.0%,0.0%,0.0%
cluster_active_noise,15.4%,5.1%,46.5%,6.9%,3.8%,0.6%,0.0%
cluster_active_1,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
cluster_active_2,0.0%,0.0%,0.0%,0.0%,0.1%,0.0%,0.0%
cluster_active_3,76.6%,0.7%,21.7%,0.0%,1.4%,0.0%,0.0%
cluster_active_4,0.0%,0.0%,0.1%,0.0%,0.0%,0.0%,0.0%
cluster_active_5,0.0%,0.1%,98.9%,0.1%,0.0%,0.0%,0.0%
cluster_at_risk_0,25.4%,4.3%,41.7%,0.8%,1.1%,0.1%,0.0%
cluster_at_risk_1,20.2%,4.6%,66.9%,2.0%,1.7%,0.1%,0.0%


In [30]:
# ── API usage and journey signals per cluster ──

display(Markdown('### API Usage & Journey Signals by Cluster'))

api_cluster_sql = f"""
SELECT
    cluster_label,
    {cluster_group_case} AS cluster_group,
    COUNT(*) AS developers,
    ROUND(AVG(CASE WHEN lifetime_api_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_has_api,
    ROUND(AVG(lifetime_api_count), 2) AS avg_api_count,
    ROUND(AVG(lifetime_activity_count), 2) AS avg_total_activity,
    ROUND(AVG(lifetime_discover_count), 2) AS avg_discover,
    ROUND(AVG(lifetime_learn_count), 2) AS avg_learn,
    ROUND(AVG(lifetime_evaluate_count), 2) AS avg_evaluate,
    ROUND(AVG(lifetime_build_count), 2) AS avg_build,
    ROUND(AVG(lifetime_champion_count), 2) AS avg_champion,
    ROUND(AVG(lifetime_high_effort_count), 2) AS avg_high_effort
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1, 2
ORDER BY cluster_group, developers DESC
"""

api_cluster = con.execute(api_cluster_sql).fetchdf()
display(api_cluster)

# ── Same view at lifecycle group level ──
display(Markdown('### API Usage & Journey Signals by Lifecycle Group'))

api_group_sql = f"""
SELECT
    {cluster_group_case} AS cluster_group,
    COUNT(*) AS developers,
    ROUND(AVG(CASE WHEN lifetime_api_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_has_api,
    ROUND(AVG(lifetime_api_count), 2) AS avg_api_count,
    ROUND(AVG(lifetime_activity_count), 2) AS avg_total_activity,
    ROUND(AVG(lifetime_discover_count), 2) AS avg_discover,
    ROUND(AVG(lifetime_learn_count), 2) AS avg_learn,
    ROUND(AVG(lifetime_evaluate_count), 2) AS avg_evaluate,
    ROUND(AVG(lifetime_build_count), 2) AS avg_build,
    ROUND(AVG(lifetime_champion_count), 2) AS avg_champion,
    ROUND(AVG(lifetime_high_effort_count), 2) AS avg_high_effort
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1
ORDER BY 1
"""

api_group = con.execute(api_group_sql).fetchdf()
display(api_group)

### API Usage & Journey Signals by Cluster

,cluster_label,cluster_group,developers,pct_has_api,avg_api_count,avg_total_activity,avg_discover,avg_learn,avg_evaluate,avg_build,avg_champion,avg_high_effort
0,cluster_active_0,active,155026,0.0019,0.00,1.03,1.00,0.03,0.00,0.00,0.00,1.00
1,cluster_active_noise,active,105723,0.3146,30.87,116.73,33.75,1.66,5.55,69.61,6.16,46.81
2,cluster_active_1,active,66683,0.9581,11.76,12.83,12.83,0.00,0.00,0.00,0.00,1.01
3,cluster_active_2,active,29395,0.9990,37.02,38.03,38.03,0.00,0.00,0.00,0.00,1.00
4,cluster_active_3,active,24653,0.0007,0.00,2.20,1.06,0.89,0.23,0.00,0.02,1.20
5,cluster_active_4,active,18554,0.0008,0.00,1.04,1.04,0.00,0.00,0.00,0.00,1.00
6,cluster_active_5,active,18015,0.0026,0.00,2.31,1.10,0.00,0.07,1.15,0.00,1.02
7,cluster_at_risk_0,at_risk,436295,0.0211,0.07,9.65,1.45,0.60,1.37,6.12,0.12,1.16
8,cluster_at_risk_1,at_risk,365945,0.0488,0.31,17.46,1.96,1.53,2.56,11.31,0.09,1.11
9,cluster_at_risk_2,at_risk,210377,0.0136,0.01,2.00,1.06,0.80,0.14,0.00,0.00,1.00


### API Usage & Journey Signals by Lifecycle Group

,cluster_group,developers,pct_has_api,avg_api_count,avg_total_activity,avg_discover,avg_learn,avg_evaluate,avg_build,avg_champion,avg_high_effort
0,active,418049,0.3035,12.29,34.90,13.78,0.48,1.42,17.65,1.56,12.60
1,at_risk,1580877,0.0271,0.43,9.35,1.82,1.00,1.23,5.18,0.13,1.52
2,cooling,356500,0.0986,1.10,13.44,2.61,0.76,1.76,7.98,0.32,2.09


In [31]:
# Re-open connection if needed (safe to run if already open)
import duckdb, pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 100)

DB_PATH = 'developer_project.duckdb'
try:
    con.execute("SELECT 1")
except Exception:
    con = duckdb.connect(DB_PATH)

PROFILE_TABLE = 'cluster_profile_asset_base_v2'

# Confirm columns we need exist
schema = con.execute(f"DESCRIBE {PROFILE_TABLE}").fetchdf()
available_cols = set(schema['column_name'].astype(str))

has_api = 'lifetime_api_count' in available_cols
has_velocity = 'activity_velocity_0_30_vs_30_90' in available_cols

print(f"Table: {PROFILE_TABLE}")
print(f"Has lifetime_api_count: {has_api}")
print(f"Has activity_velocity: {has_velocity}")
print(f"Total columns: {len(available_cols)}")

Table: cluster_profile_asset_base_v2
Has lifetime_api_count: True
Has activity_velocity: True
Total columns: 148


### 1. Exposed vs Unexposed: Pre/Post Deltas by Lifecycle Group

For each asset type (training, webinar, download), compare exposed vs unexposed developers within each lifecycle group. Uses 30-90d as baseline and 0-30d as recent window.

In [32]:
# ── Cluster group case expression (reuse from pipeline) ──
cluster_group_case = """
CASE
    WHEN cluster_label LIKE 'cluster_active_%' THEN 'active'
    WHEN cluster_label LIKE 'cluster_cooling_%' THEN 'cooling'
    WHEN cluster_label LIKE 'cluster_at_risk_%' THEN 'at_risk'
    ELSE NULL
END
"""

api_avg = "AVG(lifetime_api_count) AS avg_lifetime_api_count," if has_api else ""
velocity_avg = "AVG(activity_velocity_0_30_vs_30_90) AS avg_activity_velocity," if has_velocity else ""

# ── Training: exposed vs unexposed by lifecycle group ──
display(Markdown('### Training Exposure: Deltas by Lifecycle Group'))

training_delta_sql = f"""
SELECT
    {cluster_group_case} AS cluster_group,
    CASE WHEN lifetime_dli_training_count > 0 THEN 'Training Exposed' ELSE 'Not Exposed' END AS exposure,
    COUNT(*) AS developers,
    AVG(activity_count_30_90d) AS avg_activity_30_90d,
    AVG(activity_count_0_30d) AS avg_activity_0_30d,
    AVG(activity_count_0_30d - activity_count_30_90d) AS delta_activity,
    AVG(high_effort_count_30_90d) AS avg_high_effort_30_90d,
    AVG(high_effort_count_0_30d) AS avg_high_effort_0_30d,
    AVG(high_effort_count_0_30d - high_effort_count_30_90d) AS delta_high_effort,
    {api_avg}
    {velocity_avg}
    AVG(weighted_recent_activity) AS avg_weighted_recent_activity
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 2
"""

training_delta = con.execute(training_delta_sql).fetchdf()
display(training_delta)

# ── Webinar: exposed vs unexposed by lifecycle group ──
display(Markdown('### Webinar Exposure: Deltas by Lifecycle Group'))

webinar_delta_sql = f"""
SELECT
    {cluster_group_case} AS cluster_group,
    CASE WHEN lifetime_webinar_count > 0 THEN 'Webinar Exposed' ELSE 'Not Exposed' END AS exposure,
    COUNT(*) AS developers,
    AVG(activity_count_30_90d) AS avg_activity_30_90d,
    AVG(activity_count_0_30d) AS avg_activity_0_30d,
    AVG(activity_count_0_30d - activity_count_30_90d) AS delta_activity,
    AVG(high_effort_count_30_90d) AS avg_high_effort_30_90d,
    AVG(high_effort_count_0_30d) AS avg_high_effort_0_30d,
    AVG(high_effort_count_0_30d - high_effort_count_30_90d) AS delta_high_effort,
    {api_avg}
    {velocity_avg}
    AVG(weighted_recent_activity) AS avg_weighted_recent_activity
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 2
"""

webinar_delta = con.execute(webinar_delta_sql).fetchdf()
display(webinar_delta)

# ── Download: exposed vs unexposed by lifecycle group ──
display(Markdown('### Download Exposure: Deltas by Lifecycle Group'))

download_delta_sql = f"""
SELECT
    {cluster_group_case} AS cluster_group,
    CASE WHEN (lifetime_devzone_download_count + lifetime_ngc_download_count) > 0
         THEN 'Download Exposed' ELSE 'Not Exposed' END AS exposure,
    COUNT(*) AS developers,
    AVG(activity_count_30_90d) AS avg_activity_30_90d,
    AVG(activity_count_0_30d) AS avg_activity_0_30d,
    AVG(activity_count_0_30d - activity_count_30_90d) AS delta_activity,
    AVG(high_effort_count_30_90d) AS avg_high_effort_30_90d,
    AVG(high_effort_count_0_30d) AS avg_high_effort_0_30d,
    AVG(high_effort_count_0_30d - high_effort_count_30_90d) AS delta_high_effort,
    {api_avg}
    {velocity_avg}
    AVG(weighted_recent_activity) AS avg_weighted_recent_activity
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 2
"""

download_delta = con.execute(download_delta_sql).fetchdf()
display(download_delta)

### Training Exposure: Deltas by Lifecycle Group

,cluster_group,exposure,developers,avg_activity_30_90d,avg_activity_0_30d,delta_activity,avg_high_effort_30_90d,avg_high_effort_0_30d,delta_high_effort,avg_lifetime_api_count,avg_activity_velocity,avg_weighted_recent_activity
0,active,Not Exposed,381834,5.477467,7.883379,2.405912,1.055286,1.302377,0.247092,12.696955,3.097577,6.754562
1,active,Training Exposed,36215,3.796631,5.079580,1.282949,0.836559,1.349330,0.512771,7.954328,1.868369,4.653798
2,at_risk,Not Exposed,1196041,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.516801,NaN,0.114240
3,at_risk,Training Exposed,384836,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.168038,NaN,0.090336
4,cooling,Not Exposed,290951,3.853979,0.000000,-3.853979,0.780833,0.000000,-0.780833,1.197233,0.000000,1.285557
5,cooling,Training Exposed,65549,3.100322,0.000000,-3.100322,0.953714,0.000000,-0.953714,0.686158,0.000000,1.050764


### Webinar Exposure: Deltas by Lifecycle Group

,cluster_group,exposure,developers,avg_activity_30_90d,avg_activity_0_30d,delta_activity,avg_high_effort_30_90d,avg_high_effort_0_30d,delta_high_effort,avg_lifetime_api_count,avg_activity_velocity,avg_weighted_recent_activity
0,active,Not Exposed,410988,5.304897,7.645369,2.340472,1.022015,1.281298,0.259283,12.240226,3.003284,6.558173
1,active,Webinar Exposed,7061,6.901147,7.356465,0.455318,1.869990,2.770146,0.900156,14.956663,2.026789,7.410877
2,at_risk,Not Exposed,1522295,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.433913,NaN,0.108424
3,at_risk,Webinar Exposed,58582,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.379622,NaN,0.108342
4,cooling,Not Exposed,341429,3.755249,0.000000,-3.755249,0.834674,0.000000,-0.834674,1.113874,0.000000,1.250598
5,cooling,Webinar Exposed,15071,2.812753,0.000000,-2.812753,0.312985,0.000000,-0.312985,0.862849,0.000000,1.056347


### Download Exposure: Deltas by Lifecycle Group

,cluster_group,exposure,developers,avg_activity_30_90d,avg_activity_0_30d,delta_activity,avg_high_effort_30_90d,avg_high_effort_0_30d,delta_high_effort,avg_lifetime_api_count,avg_activity_velocity,avg_weighted_recent_activity
0,active,Download Exposed,75373,9.787935,11.682419,1.894485,5.179454,3.428814,-1.750640,7.155016,2.635621,11.087733
1,active,Not Exposed,342676,4.351726,6.751450,2.399725,0.125042,0.839621,0.714579,13.414712,3.092325,5.579448
2,at_risk,Download Exposed,666321,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.151505,NaN,0.176551
3,at_risk,Not Exposed,914556,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.636190,NaN,0.058783
4,cooling,Download Exposed,123239,6.840343,0.000000,-6.840343,0.746744,0.000000,-0.746744,0.661000,0.000000,2.329367
5,cooling,Not Exposed,233261,2.064404,0.000000,-2.064404,0.847424,0.000000,-0.847424,1.336923,0.000000,0.668101


## Pipeline Post-Visualization Sections

These sections were moved from the HDBSCAN pipeline so the pipeline can stay focused on table-building while this notebook holds the follow-on analysis and recommendation views.


In [ ]:
cluster_order_df = con.execute(f"SELECT cluster_label, developers FROM {CLUSTER_ASSET_SUMMARY_TABLE} ORDER BY developers DESC").fetchdf()
cluster_order = cluster_order_df['cluster_label'].tolist()

def finish_axes(ax, title, xlabel='', ylabel=''):
    ax.set_title(title, fontsize=11, weight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

def apply_plain_number_format(ax, axis='x'):
    fmt = StrMethodFormatter('{x:,.0f}')
    if axis == 'x':
        ax.xaxis.set_major_formatter(fmt)
    else:
        ax.yaxis.set_major_formatter(fmt)


## 11b. API Usage & Journey Signal Visualizations

These charts show API adoption and journey signal composition (Discover, Learn, Evaluate, Build, Champion) per cluster. The journey signal counts capture **all** activity types including SDK downloads, API calls, container pulls, etc. — not just the 7 named assets tracked above.

In [ ]:
# ============================================================
# API USAGE & JOURNEY SIGNAL VISUALIZATIONS
# ============================================================

cluster_group_case = """
CASE
    WHEN cluster_label LIKE 'cluster_active_%' THEN 'active'
    WHEN cluster_label LIKE 'cluster_cooling_%' THEN 'cooling'
    WHEN cluster_label LIKE 'cluster_at_risk_%' THEN 'at_risk'
    ELSE NULL
END
"""

# ── Pull API and journey signal data per cluster ──
api_journey_sql = f"""
SELECT
    cluster_label,
    {cluster_group_case} AS cluster_group,
    COUNT(*) AS developers,
    AVG(CASE WHEN lifetime_api_count > 0 THEN 1.0 ELSE 0.0 END) AS pct_has_api,
    AVG(lifetime_api_count) AS avg_api_count,
    AVG(lifetime_activity_count) AS avg_total_activity,
    AVG(lifetime_discover_count) AS avg_discover,
    AVG(lifetime_learn_count) AS avg_learn,
    AVG(lifetime_evaluate_count) AS avg_evaluate,
    AVG(lifetime_build_count) AS avg_build,
    AVG(lifetime_champion_count) AS avg_champion,
    AVG(lifetime_high_effort_count) AS avg_high_effort
FROM {CLUSTER_PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1, 2
ORDER BY cluster_group, developers DESC
"""

api_journey_df = con.execute(api_journey_sql).fetchdf()
api_journey_df['cluster_label'] = pd.Categorical(
    api_journey_df['cluster_label'], categories=cluster_order, ordered=True
)
api_journey_df = api_journey_df.sort_values('cluster_label')

# ── Same at lifecycle group level ──
api_group_sql = f"""
SELECT
    {cluster_group_case} AS cluster_group,
    COUNT(*) AS developers,
    AVG(CASE WHEN lifetime_api_count > 0 THEN 1.0 ELSE 0.0 END) AS pct_has_api,
    AVG(lifetime_api_count) AS avg_api_count,
    AVG(lifetime_activity_count) AS avg_total_activity,
    AVG(lifetime_discover_count) AS avg_discover,
    AVG(lifetime_learn_count) AS avg_learn,
    AVG(lifetime_evaluate_count) AS avg_evaluate,
    AVG(lifetime_build_count) AS avg_build,
    AVG(lifetime_champion_count) AS avg_champion
FROM {CLUSTER_PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1
ORDER BY 1
"""

api_group_df = con.execute(api_group_sql).fetchdf()

# ── Chart 1: API Adoption Rate by Cluster ──
display(Markdown('### API Adoption Rate By Cluster'))

fig, ax = plt.subplots(figsize=(16, 7))
colors = ['#264653' if 'active' in str(g) else '#e76f51' if 'at_risk' in str(g) else '#2a9d8f'
          for g in api_journey_df['cluster_group']]
ax.bar(api_journey_df['cluster_label'].astype(str), api_journey_df['pct_has_api'], color=colors)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
finish_axes(ax, 'Share Of Developers With Any API Usage', ylabel='% With API Usage')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# ── Chart 2: Average API Count by Cluster ──
display(Markdown('### Average API Count By Cluster'))

fig, ax = plt.subplots(figsize=(16, 7))
ax.bar(api_journey_df['cluster_label'].astype(str), api_journey_df['avg_api_count'], color=colors)
apply_plain_number_format(ax, axis='y')
finish_axes(ax, 'Average Lifetime API Count Per Developer', ylabel='Avg API Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# ── Chart 3: Journey Signal Composition by Cluster (stacked bar) ──
display(Markdown('### Journey Signal Composition By Cluster'))

journey_cols = ['avg_discover', 'avg_learn', 'avg_evaluate', 'avg_build', 'avg_champion']
journey_labels = ['Discover', 'Learn', 'Evaluate', 'Build', 'Champion']
journey_colors = ['#e9c46a', '#2a9d8f', '#264653', '#e76f51', '#f4a261']

fig, ax = plt.subplots(figsize=(16, 7))
plot_data = api_journey_df.set_index('cluster_label')[journey_cols].copy()
plot_data.columns = journey_labels
plot_data.plot(kind='bar', stacked=True, ax=ax, color=journey_colors)
apply_plain_number_format(ax, axis='y')
finish_axes(ax, 'Average Journey Signal Counts By Cluster', ylabel='Avg Activity Count')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Journey Signal', bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
plt.tight_layout()
plt.show()

# ── Chart 4: Journey Signal Composition by Lifecycle Group (stacked bar) ──
display(Markdown('### Journey Signal Composition By Lifecycle Group'))

fig, ax = plt.subplots(figsize=(10, 6))
group_plot = api_group_df.set_index('cluster_group')[journey_cols].copy()
group_plot.columns = journey_labels
group_plot.plot(kind='bar', stacked=True, ax=ax, color=journey_colors)
apply_plain_number_format(ax, axis='y')
finish_axes(ax, 'Average Journey Signal Counts By Lifecycle Group', ylabel='Avg Activity Count')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Journey Signal', bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
plt.tight_layout()
plt.show()

# ── Chart 5: API Adoption Rate by Lifecycle Group ──
display(Markdown('### API Adoption By Lifecycle Group'))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
group_colors = ['#264653', '#e76f51', '#2a9d8f']

axes[0].bar(api_group_df['cluster_group'], api_group_df['pct_has_api'], color=group_colors)
axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
finish_axes(axes[0], 'Share With Any API Usage', ylabel='% With API')

axes[1].bar(api_group_df['cluster_group'], api_group_df['avg_api_count'], color=group_colors)
apply_plain_number_format(axes[1], axis='y')
finish_axes(axes[1], 'Average Lifetime API Count', ylabel='Avg API Count')

plt.tight_layout()
plt.show()

# ── Table: Full API & Journey Data ──
display(Markdown('### API & Journey Signal Summary Table'))
display(api_journey_df)
display(api_group_df)

## 12. Persona And Asset Correlation Tables

This section adds ranked tables to help answer which personas dominate each cluster and which activity asset types are most associated with those personas.

It also creates a rolled-up view for the main HDBSCAN lifecycle groups: `active`, `cooling`, and `at_risk`, split by persona, effort, and journey stage.

In [ ]:
CLUSTER_PERSONA_RANK_TABLE = 'cluster_persona_rank_v2'
CLUSTER_PERSONA_ASSET_RANK_TABLE = 'cluster_persona_asset_rank_v2'
CLUSTER_GROUP_PERSONA_ASSET_RANK_TABLE = 'cluster_group_persona_effort_journey_asset_rank_v2'
CLUSTER_SEGMENT_ASSET_RANK_TABLE = 'cluster_persona_effort_journey_asset_rank_v2'
CLUSTER_TOP_PERSONA_SUMMARY_TABLE = 'cluster_top_persona_summary_v2'
CLUSTER_TOP_ASSET_SUMMARY_TABLE = 'cluster_top_asset_summary_v2'
CLUSTER_CORRELATION_SUMMARY_TABLE = 'cluster_correlation_summary_v2'
CLUSTER_GROUP_ROLLUP_TABLE = 'cluster_group_rollup_summary_v2'

cluster_group_case = """
CASE
    WHEN cluster_label LIKE 'cluster_active_%' THEN 'active'
    WHEN cluster_label LIKE 'cluster_cooling_%' THEN 'cooling'
    WHEN cluster_label LIKE 'cluster_at_risk_%' THEN 'at_risk'
    ELSE NULL
END
"""
cluster_group_case_b = cluster_group_case.replace('cluster_label', 'b.cluster_label')

persona_rank_sql = f"""
CREATE OR REPLACE TABLE {CLUSTER_PERSONA_RANK_TABLE} AS
WITH persona_counts AS (
    SELECT
        cluster_label,
        persona,
        COUNT(*) AS developers
    FROM {CLUSTER_PROFILE_TABLE}
    GROUP BY 1,2
)
SELECT
    cluster_label,
    persona,
    developers,
    developers * 1.0 / NULLIF(SUM(developers) OVER (PARTITION BY cluster_label), 0) AS pct_within_cluster,
    ROW_NUMBER() OVER (
        PARTITION BY cluster_label
        ORDER BY developers DESC, persona
    ) AS rank_within_cluster
FROM persona_counts
"""

cluster_persona_asset_rank_sql = f"""
CREATE OR REPLACE TABLE {CLUSTER_PERSONA_ASSET_RANK_TABLE} AS
WITH persona_cluster_sizes AS (
    SELECT
        cluster_label,
        persona,
        COUNT(*) AS persona_cluster_developers
    FROM {CLUSTER_PROFILE_TABLE}
    GROUP BY 1,2
),
persona_asset_stats AS (
    SELECT
        b.cluster_label,
        b.persona,
        l.asset_name,
        l.asset_group,
        l.asset_role,
        l.signal_type,
        COUNT(*) AS exposed_developers,
        AVG(l.exposure_value) AS avg_exposure_value_exposed,
        MEDIAN(l.exposure_value) AS median_exposure_value_exposed,
        SUM(l.exposure_value) AS total_exposure_value
    FROM {CLUSTER_ASSET_LONG_TABLE} l
    JOIN {CLUSTER_PROFILE_TABLE} b
      ON l.developer_id = b.developer_id
     AND l.cluster_id = b.cluster_id
     AND l.cluster_label = b.cluster_label
    WHERE l.analysis_role = 'rankable_activity_asset'
      AND l.exposed_flag = 1
    GROUP BY 1,2,3,4,5,6
)
SELECT
    s.cluster_label,
    s.persona,
    p.persona_cluster_developers,
    s.asset_name,
    s.asset_group,
    s.asset_role,
    s.signal_type,
    s.exposed_developers,
    s.exposed_developers * 1.0 / NULLIF(p.persona_cluster_developers, 0) AS exposure_rate_within_persona_cluster,
    s.avg_exposure_value_exposed,
    s.median_exposure_value_exposed,
    s.total_exposure_value,
    s.total_exposure_value * 1.0 / NULLIF(p.persona_cluster_developers, 0) AS asset_volume_per_persona_cluster_developer,
    ROW_NUMBER() OVER (
        PARTITION BY s.cluster_label, s.persona
        ORDER BY s.total_exposure_value * 1.0 / NULLIF(p.persona_cluster_developers, 0) DESC,
                 s.exposed_developers DESC,
                 s.avg_exposure_value_exposed DESC,
                 s.asset_name
    ) AS rank_by_total_volume,
    ROW_NUMBER() OVER (
        PARTITION BY s.cluster_label, s.persona
        ORDER BY s.exposed_developers * 1.0 / NULLIF(p.persona_cluster_developers, 0) DESC,
                 s.exposed_developers DESC,
                 s.asset_name
    ) AS rank_by_breadth,
    ROW_NUMBER() OVER (
        PARTITION BY s.cluster_label, s.persona
        ORDER BY s.avg_exposure_value_exposed DESC,
                 s.exposed_developers DESC,
                 s.asset_name
    ) AS rank_by_intensity
FROM persona_asset_stats s
JOIN persona_cluster_sizes p
  ON s.cluster_label = p.cluster_label
 AND s.persona = p.persona
"""

cluster_group_persona_asset_rank_sql = f"""
CREATE OR REPLACE TABLE {CLUSTER_GROUP_PERSONA_ASSET_RANK_TABLE} AS
WITH profile_grouped AS (
    SELECT
        developer_id,
        cluster_id,
        cluster_label,
        {cluster_group_case} AS cluster_group,
        persona,
        developer_effort_level,
        behavior_journey_stage_30d
    FROM {CLUSTER_PROFILE_TABLE}
),
segment_sizes AS (
    SELECT
        cluster_group,
        persona,
        developer_effort_level,
        behavior_journey_stage_30d,
        COUNT(*) AS segment_developers
    FROM profile_grouped
    WHERE cluster_group IS NOT NULL
    GROUP BY 1,2,3,4
),
segment_asset_stats AS (
    SELECT
        p.cluster_group,
        p.persona,
        p.developer_effort_level,
        p.behavior_journey_stage_30d,
        l.asset_name,
        l.asset_group,
        l.asset_role,
        l.signal_type,
        COUNT(*) AS exposed_developers,
        AVG(l.exposure_value) AS avg_exposure_value_exposed,
        MEDIAN(l.exposure_value) AS median_exposure_value_exposed,
        SUM(l.exposure_value) AS total_exposure_value
    FROM {CLUSTER_ASSET_LONG_TABLE} l
    JOIN profile_grouped p
      ON l.developer_id = p.developer_id
     AND l.cluster_id = p.cluster_id
     AND l.cluster_label = p.cluster_label
    WHERE p.cluster_group IS NOT NULL
      AND l.analysis_role = 'rankable_activity_asset'
      AND l.exposed_flag = 1
    GROUP BY 1,2,3,4,5,6,7,8
)
SELECT
    s.cluster_group,
    s.persona,
    s.developer_effort_level,
    s.behavior_journey_stage_30d,
    z.segment_developers,
    s.asset_name,
    s.asset_group,
    s.asset_role,
    s.signal_type,
    s.exposed_developers,
    s.exposed_developers * 1.0 / NULLIF(z.segment_developers, 0) AS exposure_rate_within_segment,
    s.avg_exposure_value_exposed,
    s.median_exposure_value_exposed,
    s.total_exposure_value,
    s.total_exposure_value * 1.0 / NULLIF(z.segment_developers, 0) AS asset_volume_per_segment_developer,
    ROW_NUMBER() OVER (
        PARTITION BY s.cluster_group, s.persona, s.developer_effort_level, s.behavior_journey_stage_30d
        ORDER BY s.total_exposure_value * 1.0 / NULLIF(z.segment_developers, 0) DESC,
                 s.exposed_developers DESC,
                 s.avg_exposure_value_exposed DESC,
                 s.asset_name
    ) AS rank_by_total_volume,
    ROW_NUMBER() OVER (
        PARTITION BY s.cluster_group, s.persona, s.developer_effort_level, s.behavior_journey_stage_30d
        ORDER BY s.exposed_developers * 1.0 / NULLIF(z.segment_developers, 0) DESC,
                 s.exposed_developers DESC,
                 s.asset_name
    ) AS rank_by_breadth,
    ROW_NUMBER() OVER (
        PARTITION BY s.cluster_group, s.persona, s.developer_effort_level, s.behavior_journey_stage_30d
        ORDER BY s.avg_exposure_value_exposed DESC,
                 s.exposed_developers DESC,
                 s.asset_name
    ) AS rank_by_intensity
FROM segment_asset_stats s
JOIN segment_sizes z
  ON s.cluster_group = z.cluster_group
 AND s.persona = z.persona
 AND s.developer_effort_level = z.developer_effort_level
 AND s.behavior_journey_stage_30d = z.behavior_journey_stage_30d
"""

cluster_segment_asset_rank_sql = f"""
CREATE OR REPLACE TABLE {CLUSTER_SEGMENT_ASSET_RANK_TABLE} AS
WITH segment_sizes AS (
    SELECT
        cluster_label,
        {cluster_group_case} AS cluster_group,
        persona,
        developer_effort_level,
        behavior_journey_stage_30d,
        COUNT(*) AS segment_developers
    FROM {CLUSTER_PROFILE_TABLE}
    GROUP BY 1,2,3,4,5
),
segment_asset_stats AS (
    SELECT
        b.cluster_label,
        {cluster_group_case_b} AS cluster_group,
        b.persona,
        b.developer_effort_level,
        b.behavior_journey_stage_30d,
        l.asset_name,
        l.asset_group,
        l.asset_role,
        l.signal_type,
        COUNT(*) AS exposed_developers,
        AVG(l.exposure_value) AS avg_exposure_value_exposed,
        MEDIAN(l.exposure_value) AS median_exposure_value_exposed,
        SUM(l.exposure_value) AS total_exposure_value
    FROM {CLUSTER_ASSET_LONG_TABLE} l
    JOIN {CLUSTER_PROFILE_TABLE} b
      ON l.developer_id = b.developer_id
     AND l.cluster_id = b.cluster_id
     AND l.cluster_label = b.cluster_label
    WHERE l.analysis_role = 'rankable_activity_asset'
      AND l.exposed_flag = 1
    GROUP BY 1,2,3,4,5,6,7,8,9
)
SELECT
    s.cluster_label,
    s.cluster_group,
    s.persona,
    s.developer_effort_level,
    s.behavior_journey_stage_30d,
    z.segment_developers,
    s.asset_name,
    s.asset_group,
    s.asset_role,
    s.signal_type,
    s.exposed_developers,
    s.exposed_developers * 1.0 / NULLIF(z.segment_developers, 0) AS exposure_rate_within_segment,
    s.avg_exposure_value_exposed,
    s.median_exposure_value_exposed,
    s.total_exposure_value,
    s.total_exposure_value * 1.0 / NULLIF(z.segment_developers, 0) AS asset_volume_per_segment_developer,
    ROW_NUMBER() OVER (
        PARTITION BY s.cluster_label, s.persona, s.developer_effort_level, s.behavior_journey_stage_30d
        ORDER BY s.total_exposure_value * 1.0 / NULLIF(z.segment_developers, 0) DESC,
                 s.exposed_developers DESC,
                 s.avg_exposure_value_exposed DESC,
                 s.asset_name
    ) AS rank_by_total_volume,
    ROW_NUMBER() OVER (
        PARTITION BY s.cluster_label, s.persona, s.developer_effort_level, s.behavior_journey_stage_30d
        ORDER BY s.exposed_developers * 1.0 / NULLIF(z.segment_developers, 0) DESC,
                 s.exposed_developers DESC,
                 s.asset_name
    ) AS rank_by_breadth,
    ROW_NUMBER() OVER (
        PARTITION BY s.cluster_label, s.persona, s.developer_effort_level, s.behavior_journey_stage_30d
        ORDER BY s.avg_exposure_value_exposed DESC,
                 s.exposed_developers DESC,
                 s.asset_name
    ) AS rank_by_intensity
FROM segment_asset_stats s
JOIN segment_sizes z
  ON s.cluster_label = z.cluster_label
 AND COALESCE(s.cluster_group, '') = COALESCE(z.cluster_group, '')
 AND s.persona = z.persona
 AND s.developer_effort_level = z.developer_effort_level
 AND s.behavior_journey_stage_30d = z.behavior_journey_stage_30d
"""

top_persona_summary_sql = f"""
CREATE OR REPLACE TABLE {CLUSTER_TOP_PERSONA_SUMMARY_TABLE} AS
SELECT
    cluster_label,
    MAX(CASE WHEN rank_within_cluster = 1 THEN persona END) AS top_persona_1,
    MAX(CASE WHEN rank_within_cluster = 1 THEN pct_within_cluster END) AS top_persona_1_share,
    MAX(CASE WHEN rank_within_cluster = 2 THEN persona END) AS top_persona_2,
    MAX(CASE WHEN rank_within_cluster = 2 THEN pct_within_cluster END) AS top_persona_2_share,
    MAX(CASE WHEN rank_within_cluster = 3 THEN persona END) AS top_persona_3,
    MAX(CASE WHEN rank_within_cluster = 3 THEN pct_within_cluster END) AS top_persona_3_share
FROM {CLUSTER_PERSONA_RANK_TABLE}
GROUP BY 1
"""

top_asset_summary_sql = f"""
CREATE OR REPLACE TABLE {CLUSTER_TOP_ASSET_SUMMARY_TABLE} AS
SELECT
    cluster_label,
    MAX(CASE WHEN rank_by_total_volume = 1 THEN asset_name END) AS top_volume_asset_1,
    MAX(CASE WHEN rank_by_total_volume = 2 THEN asset_name END) AS top_volume_asset_2,
    MAX(CASE WHEN rank_by_total_volume = 3 THEN asset_name END) AS top_volume_asset_3,
    MAX(CASE WHEN rank_by_breadth = 1 THEN asset_name END) AS top_breadth_asset_1,
    MAX(CASE WHEN rank_by_breadth = 2 THEN asset_name END) AS top_breadth_asset_2,
    MAX(CASE WHEN rank_by_breadth = 3 THEN asset_name END) AS top_breadth_asset_3,
    MAX(CASE WHEN rank_by_intensity = 1 THEN asset_name END) AS top_intensity_asset_1,
    MAX(CASE WHEN rank_by_intensity = 2 THEN asset_name END) AS top_intensity_asset_2,
    MAX(CASE WHEN rank_by_intensity = 3 THEN asset_name END) AS top_intensity_asset_3
FROM {CLUSTER_ASSET_PRIORITY_TABLE}
GROUP BY 1
"""

correlation_summary_sql = f"""
CREATE OR REPLACE TABLE {CLUSTER_CORRELATION_SUMMARY_TABLE} AS
WITH cluster_sizes AS (
    SELECT
        cluster_label,
        {cluster_group_case} AS cluster_group,
        COUNT(*) AS cluster_developers
    FROM {CLUSTER_PROFILE_TABLE}
    GROUP BY 1,2
),
lead_effort AS (
    SELECT cluster_label, effort_bucket AS dominant_effort, pct_within_cluster AS dominant_effort_share
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cluster_label ORDER BY developers DESC, effort_bucket) AS rn
        FROM {CLUSTER_EFFORT_PROFILE_TABLE}
    )
    WHERE rn = 1
),
lead_journey AS (
    SELECT cluster_label, journey_bucket AS dominant_journey, pct_within_cluster AS dominant_journey_share
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cluster_label ORDER BY developers DESC, journey_bucket) AS rn
        FROM {CLUSTER_JOURNEY_PROFILE_TABLE}
    )
    WHERE rn = 1
),
lead_segment AS (
    SELECT
        cluster_label,
        cluster_group,
        persona AS dominant_segment_persona,
        developer_effort_level AS dominant_segment_effort,
        behavior_journey_stage_30d AS dominant_segment_journey,
        segment_developers AS dominant_segment_developers,
        segment_developers * 1.0 / NULLIF(SUM(segment_developers) OVER (PARTITION BY cluster_label), 0) AS dominant_segment_share
    FROM (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY cluster_label
            ORDER BY segment_developers DESC, persona, developer_effort_level, behavior_journey_stage_30d
        ) AS rn
        FROM (
            SELECT DISTINCT
                cluster_label,
                cluster_group,
                persona,
                developer_effort_level,
                behavior_journey_stage_30d,
                segment_developers
            FROM {CLUSTER_SEGMENT_ASSET_RANK_TABLE}
        )
    )
    WHERE rn = 1
),
lead_segment_assets AS (
    SELECT
        s.cluster_label,
        MAX(CASE WHEN s.rank_by_total_volume = 1 THEN s.asset_name END) AS dominant_segment_top_volume_asset,
        MAX(CASE WHEN s.rank_by_breadth = 1 THEN s.asset_name END) AS dominant_segment_top_breadth_asset,
        MAX(CASE WHEN s.rank_by_intensity = 1 THEN s.asset_name END) AS dominant_segment_top_intensity_asset
    FROM {CLUSTER_SEGMENT_ASSET_RANK_TABLE} s
    JOIN lead_segment l
      ON s.cluster_label = l.cluster_label
     AND COALESCE(s.cluster_group, '') = COALESCE(l.cluster_group, '')
     AND s.persona = l.dominant_segment_persona
     AND s.developer_effort_level = l.dominant_segment_effort
     AND s.behavior_journey_stage_30d = l.dominant_segment_journey
    GROUP BY 1
)
SELECT
    c.cluster_label,
    c.cluster_group,
    c.cluster_developers,
    p.top_persona_1,
    p.top_persona_1_share,
    p.top_persona_2,
    p.top_persona_2_share,
    p.top_persona_3,
    p.top_persona_3_share,
    e.dominant_effort,
    e.dominant_effort_share,
    j.dominant_journey,
    j.dominant_journey_share,
    a.top_volume_asset_1,
    a.top_volume_asset_2,
    a.top_volume_asset_3,
    a.top_breadth_asset_1,
    a.top_breadth_asset_2,
    a.top_breadth_asset_3,
    a.top_intensity_asset_1,
    a.top_intensity_asset_2,
    a.top_intensity_asset_3,
    l.dominant_segment_persona,
    l.dominant_segment_effort,
    l.dominant_segment_journey,
    l.dominant_segment_developers,
    l.dominant_segment_share,
    sa.dominant_segment_top_volume_asset,
    sa.dominant_segment_top_breadth_asset,
    sa.dominant_segment_top_intensity_asset
FROM cluster_sizes c
LEFT JOIN {CLUSTER_TOP_PERSONA_SUMMARY_TABLE} p USING (cluster_label)
LEFT JOIN lead_effort e USING (cluster_label)
LEFT JOIN lead_journey j USING (cluster_label)
LEFT JOIN {CLUSTER_TOP_ASSET_SUMMARY_TABLE} a USING (cluster_label)
LEFT JOIN lead_segment l USING (cluster_label)
LEFT JOIN lead_segment_assets sa USING (cluster_label)
"""


group_rollup_sql = f"""
CREATE OR REPLACE TABLE {CLUSTER_GROUP_ROLLUP_TABLE} AS
WITH group_sizes AS (
    SELECT {cluster_group_case} AS cluster_group, COUNT(*) AS developers
    FROM {CLUSTER_PROFILE_TABLE}
    WHERE {cluster_group_case} IS NOT NULL
    GROUP BY 1
),
persona_rank AS (
    SELECT
        {cluster_group_case} AS cluster_group,
        persona,
        COUNT(*) AS developers,
        COUNT(*) * 1.0 / NULLIF(SUM(COUNT(*)) OVER (PARTITION BY {cluster_group_case}), 0) AS share,
        ROW_NUMBER() OVER (
            PARTITION BY {cluster_group_case}
            ORDER BY COUNT(*) DESC, persona
        ) AS rn
    FROM {CLUSTER_PROFILE_TABLE}
    WHERE {cluster_group_case} IS NOT NULL
    GROUP BY 1,2
),
effort_rank AS (
    SELECT
        {cluster_group_case} AS cluster_group,
        developer_effort_level AS effort_level,
        COUNT(*) AS developers,
        COUNT(*) * 1.0 / NULLIF(SUM(COUNT(*)) OVER (PARTITION BY {cluster_group_case}), 0) AS share,
        ROW_NUMBER() OVER (
            PARTITION BY {cluster_group_case}
            ORDER BY COUNT(*) DESC, developer_effort_level
        ) AS rn
    FROM {CLUSTER_PROFILE_TABLE}
    WHERE {cluster_group_case} IS NOT NULL
    GROUP BY 1,2
),
asset_rank AS (
    SELECT
        {cluster_group_case_b} AS cluster_group,
        l.asset_name,
        SUM(l.exposure_value) AS total_exposure_value,
        AVG(l.exposed_flag) AS exposure_rate,
        AVG(CASE WHEN l.exposed_flag = 1 THEN l.exposure_value END) AS avg_exposure_value_exposed,
        ROW_NUMBER() OVER (
            PARTITION BY {cluster_group_case_b}
            ORDER BY SUM(l.exposure_value) DESC, l.asset_name
        ) AS rank_by_volume,
        ROW_NUMBER() OVER (
            PARTITION BY {cluster_group_case_b}
            ORDER BY AVG(l.exposed_flag) DESC, SUM(l.exposure_value) DESC, l.asset_name
        ) AS rank_by_breadth,
        ROW_NUMBER() OVER (
            PARTITION BY {cluster_group_case_b}
            ORDER BY AVG(CASE WHEN l.exposed_flag = 1 THEN l.exposure_value END) DESC, SUM(l.exposure_value) DESC, l.asset_name
        ) AS rank_by_intensity
    FROM {CLUSTER_ASSET_LONG_TABLE} l
    JOIN {CLUSTER_PROFILE_TABLE} b
      ON l.developer_id = b.developer_id
     AND l.cluster_id = b.cluster_id
     AND l.cluster_label = b.cluster_label
    WHERE l.analysis_role = 'rankable_activity_asset'
      AND {cluster_group_case_b} IS NOT NULL
    GROUP BY 1,2
)
SELECT
    g.cluster_group,
    g.developers,
    MAX(CASE WHEN p.rn = 1 THEN p.persona END) AS top_persona_1,
    MAX(CASE WHEN p.rn = 1 THEN p.share END) AS top_persona_1_share,
    MAX(CASE WHEN p.rn = 2 THEN p.persona END) AS top_persona_2,
    MAX(CASE WHEN p.rn = 2 THEN p.share END) AS top_persona_2_share,
    MAX(CASE WHEN p.rn = 3 THEN p.persona END) AS top_persona_3,
    MAX(CASE WHEN p.rn = 3 THEN p.share END) AS top_persona_3_share,
    MAX(CASE WHEN e.rn = 1 THEN e.effort_level END) AS top_effort_1,
    MAX(CASE WHEN e.rn = 1 THEN e.share END) AS top_effort_1_share,
    MAX(CASE WHEN e.rn = 2 THEN e.effort_level END) AS top_effort_2,
    MAX(CASE WHEN e.rn = 2 THEN e.share END) AS top_effort_2_share,
    MAX(CASE WHEN a.rank_by_volume = 1 THEN a.asset_name END) AS top_volume_asset_1,
    MAX(CASE WHEN a.rank_by_volume = 2 THEN a.asset_name END) AS top_volume_asset_2,
    MAX(CASE WHEN a.rank_by_volume = 3 THEN a.asset_name END) AS top_volume_asset_3,
    MAX(CASE WHEN a.rank_by_breadth = 1 THEN a.asset_name END) AS top_breadth_asset_1,
    MAX(CASE WHEN a.rank_by_intensity = 1 THEN a.asset_name END) AS top_intensity_asset_1
FROM group_sizes g
LEFT JOIN persona_rank p ON g.cluster_group = p.cluster_group
LEFT JOIN effort_rank e ON g.cluster_group = e.cluster_group
LEFT JOIN asset_rank a ON g.cluster_group = a.cluster_group
GROUP BY 1,2
ORDER BY CASE g.cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
"""
con.execute(persona_rank_sql)
con.execute(cluster_persona_asset_rank_sql)
con.execute(cluster_group_persona_asset_rank_sql)
con.execute(cluster_segment_asset_rank_sql)
con.execute(top_persona_summary_sql)
con.execute(top_asset_summary_sql)
con.execute(correlation_summary_sql)
con.execute(group_rollup_sql)


display(Markdown('### Lifecycle Group Rollup Summary'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_GROUP_ROLLUP_TABLE}
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
""").fetchdf())
display(Markdown('### Top Personas By Cluster'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_TOP_PERSONA_SUMMARY_TABLE}
ORDER BY cluster_label
""").fetchdf())

display(Markdown('### Top Assets By Cluster'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_TOP_ASSET_SUMMARY_TABLE}
ORDER BY cluster_label
""").fetchdf())

display(Markdown('### Cluster Correlation Summary'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_CORRELATION_SUMMARY_TABLE}
ORDER BY cluster_group, cluster_developers DESC, cluster_label
""").fetchdf())


## 13. Differentiating Assets By Lifecycle Group

This section is meant to surface assets that are more characteristic of `active`, `cooling`, and `at_risk` than the raw top-volume tables. It uses over-index and lift so ubiquitous assets like downloads do not automatically dominate.

In [ ]:
CLUSTER_GROUP_DIFFERENTIATING_ASSET_TABLE = 'cluster_group_differentiating_assets_v2'
CLUSTER_GROUP_ASSET_WIDE_TABLE = 'cluster_group_asset_wide_summary_v2'

differentiating_asset_sql = f"""
CREATE OR REPLACE TABLE {CLUSTER_GROUP_DIFFERENTIATING_ASSET_TABLE} AS
WITH asset_group_base AS (
    SELECT
        {cluster_group_case_b} AS cluster_group,
        l.asset_name,
        l.asset_group,
        l.asset_role,
        l.signal_type,
        COUNT(*) AS developers,
        SUM(l.exposed_flag) AS exposed_developers,
        AVG(l.exposed_flag) AS exposure_rate,
        AVG(CASE WHEN l.exposed_flag = 1 THEN l.exposure_value END) AS avg_exposure_value_exposed,
        SUM(l.exposure_value) AS total_exposure_value
    FROM {CLUSTER_ASSET_LONG_TABLE} l
    JOIN {CLUSTER_PROFILE_TABLE} b
      ON l.developer_id = b.developer_id
     AND l.cluster_id = b.cluster_id
     AND l.cluster_label = b.cluster_label
    WHERE l.analysis_role = 'rankable_activity_asset'
      AND {cluster_group_case_b} IS NOT NULL
    GROUP BY 1,2,3,4,5
),
overall_asset_base AS (
    SELECT
        l.asset_name,
        l.asset_group,
        l.asset_role,
        l.signal_type,
        COUNT(*) AS developers,
        SUM(l.exposed_flag) AS exposed_developers,
        AVG(l.exposed_flag) AS exposure_rate,
        AVG(CASE WHEN l.exposed_flag = 1 THEN l.exposure_value END) AS avg_exposure_value_exposed,
        SUM(l.exposure_value) AS total_exposure_value
    FROM {CLUSTER_ASSET_LONG_TABLE} l
    WHERE l.analysis_role = 'rankable_activity_asset'
    GROUP BY 1,2,3,4
),
scored AS (
    SELECT
        g.cluster_group,
        g.asset_name,
        g.asset_group,
        g.asset_role,
        g.signal_type,
        g.developers AS group_developers,
        g.exposed_developers,
        g.exposure_rate,
        o.exposure_rate AS overall_exposure_rate,
        g.avg_exposure_value_exposed,
        o.avg_exposure_value_exposed AS overall_avg_exposure_value_exposed,
        g.total_exposure_value,
        o.total_exposure_value AS overall_total_exposure_value,
        g.exposure_rate / NULLIF(o.exposure_rate, 0) AS exposure_rate_lift,
        g.avg_exposure_value_exposed / NULLIF(o.avg_exposure_value_exposed, 0) AS intensity_lift,
        g.exposure_rate - o.exposure_rate AS exposure_rate_diff,
        g.total_exposure_value * 1.0 / NULLIF(g.developers, 0) AS asset_volume_per_group_developer,
        CASE
            WHEN o.exposure_rate >= 0.60 THEN 1
            ELSE 0
        END AS ubiquitous_asset_flag,
        ROW_NUMBER() OVER (
            PARTITION BY g.cluster_group
            ORDER BY g.exposure_rate / NULLIF(o.exposure_rate, 0) DESC,
                     g.exposure_rate DESC,
                     g.asset_name
        ) AS rank_by_lift_all_assets,
        ROW_NUMBER() OVER (
            PARTITION BY g.cluster_group
            ORDER BY CASE WHEN o.exposure_rate < 0.60 THEN g.exposure_rate / NULLIF(o.exposure_rate, 0) END DESC,
                     CASE WHEN o.exposure_rate < 0.60 THEN g.exposure_rate END DESC,
                     g.asset_name
        ) AS rank_by_lift_non_ubiquitous,
        ROW_NUMBER() OVER (
            PARTITION BY g.cluster_group
            ORDER BY CASE WHEN g.asset_role <> 'download_activity' THEN g.exposure_rate / NULLIF(o.exposure_rate, 0) END DESC,
                     CASE WHEN g.asset_role <> 'download_activity' THEN g.exposure_rate END DESC,
                     g.asset_name
        ) AS rank_by_lift_excluding_downloads
    FROM asset_group_base g
    JOIN overall_asset_base o
      ON g.asset_name = o.asset_name
     AND g.asset_group = o.asset_group
     AND g.asset_role = o.asset_role
     AND g.signal_type = o.signal_type
)
SELECT *
FROM scored
"""


asset_wide_summary_sql = f"""
CREATE OR REPLACE TABLE {CLUSTER_GROUP_ASSET_WIDE_TABLE} AS
SELECT
    cluster_group,
    MAX(group_developers) AS developers,
    MAX(CASE WHEN asset_name = 'dli_training' THEN exposure_rate END) AS pct_dli_training,
    MAX(CASE WHEN asset_name = 'dli_training' THEN exposure_rate_lift END) AS lift_dli_training,
    MAX(CASE WHEN asset_name = 'webinar' THEN exposure_rate END) AS pct_webinar,
    MAX(CASE WHEN asset_name = 'webinar' THEN exposure_rate_lift END) AS lift_webinar,
    MAX(CASE WHEN asset_name = 'forum_contribution' THEN exposure_rate END) AS pct_forum_contribution,
    MAX(CASE WHEN asset_name = 'forum_contribution' THEN exposure_rate_lift END) AS lift_forum_contribution,
    MAX(CASE WHEN asset_name = 'bug_filed' THEN exposure_rate END) AS pct_bug_filed,
    MAX(CASE WHEN asset_name = 'bug_filed' THEN exposure_rate_lift END) AS lift_bug_filed,
    MAX(CASE WHEN asset_name = 'hackathon' THEN exposure_rate END) AS pct_hackathon,
    MAX(CASE WHEN asset_name = 'hackathon' THEN exposure_rate_lift END) AS lift_hackathon,
    MAX(CASE WHEN asset_name = 'devzone_download' THEN exposure_rate END) AS pct_devzone_download,
    MAX(CASE WHEN asset_name = 'devzone_download' THEN exposure_rate_lift END) AS lift_devzone_download,
    MAX(CASE WHEN asset_name = 'ngc_download' THEN exposure_rate END) AS pct_ngc_download,
    MAX(CASE WHEN asset_name = 'ngc_download' THEN exposure_rate_lift END) AS lift_ngc_download
FROM {CLUSTER_GROUP_DIFFERENTIATING_ASSET_TABLE}
GROUP BY 1
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
"""
con.execute(differentiating_asset_sql)
con.execute(asset_wide_summary_sql)


display(Markdown('### Lifecycle Group Asset Matrix'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_GROUP_ASSET_WIDE_TABLE}
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
""").fetchdf())
display(Markdown('### Differentiating Assets By Lifecycle Group: Non-Ubiquitous'))
display(con.execute(f"""
SELECT
    cluster_group,
    rank_by_lift_non_ubiquitous,
    asset_name,
    asset_role,
    exposed_developers,
    exposure_rate,
    overall_exposure_rate,
    exposure_rate_lift,
    avg_exposure_value_exposed,
    intensity_lift,
    ubiquitous_asset_flag
FROM {CLUSTER_GROUP_DIFFERENTIATING_ASSET_TABLE}
WHERE rank_by_lift_non_ubiquitous <= 5
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END,
         rank_by_lift_non_ubiquitous
""").fetchdf())

display(Markdown('### Differentiating Assets By Lifecycle Group: Excluding Downloads'))
display(con.execute(f"""
SELECT
    cluster_group,
    rank_by_lift_excluding_downloads,
    asset_name,
    asset_role,
    exposed_developers,
    exposure_rate,
    overall_exposure_rate,
    exposure_rate_lift,
    avg_exposure_value_exposed,
    intensity_lift
FROM {CLUSTER_GROUP_DIFFERENTIATING_ASSET_TABLE}
WHERE rank_by_lift_excluding_downloads <= 5
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END,
         rank_by_lift_excluding_downloads
""").fetchdf())


## 14. Program Recommendation Tables

These tables are meant to support recommendation-making without implying causality. They summarize lifecycle-group patterns, asset audience mix, distinctive assets, and a one-row-per-cluster action mapping.

In [ ]:
CLUSTER_GROUP_ASSET_PROFILE_TABLE = 'cluster_group_asset_profile_v2'
CLUSTER_GROUP_COMPOSITION_TABLE = 'cluster_group_composition_summary_v2'
ASSET_AUDIENCE_TABLE = 'asset_audience_summary_v2'

cluster_group_asset_profile_sql = f"""
CREATE OR REPLACE TABLE {CLUSTER_GROUP_ASSET_PROFILE_TABLE} AS
SELECT
    cluster_group,
    MAX(group_developers) AS developers,
    MAX(CASE WHEN asset_name = 'dli_training' THEN exposure_rate END) AS pct_dli_training,
    MAX(CASE WHEN asset_name = 'dli_training' THEN exposure_rate_lift END) AS lift_dli_training,
    MAX(CASE WHEN asset_name = 'dli_training' THEN avg_exposure_value_exposed END) AS intensity_dli_training,
    MAX(CASE WHEN asset_name = 'webinar' THEN exposure_rate END) AS pct_webinar,
    MAX(CASE WHEN asset_name = 'webinar' THEN exposure_rate_lift END) AS lift_webinar,
    MAX(CASE WHEN asset_name = 'webinar' THEN avg_exposure_value_exposed END) AS intensity_webinar,
    MAX(CASE WHEN asset_name = 'forum_contribution' THEN exposure_rate END) AS pct_forum_contribution,
    MAX(CASE WHEN asset_name = 'forum_contribution' THEN exposure_rate_lift END) AS lift_forum_contribution,
    MAX(CASE WHEN asset_name = 'forum_contribution' THEN avg_exposure_value_exposed END) AS intensity_forum_contribution,
    MAX(CASE WHEN asset_name = 'bug_filed' THEN exposure_rate END) AS pct_bug_filed,
    MAX(CASE WHEN asset_name = 'bug_filed' THEN exposure_rate_lift END) AS lift_bug_filed,
    MAX(CASE WHEN asset_name = 'bug_filed' THEN avg_exposure_value_exposed END) AS intensity_bug_filed,
    MAX(CASE WHEN asset_name = 'hackathon' THEN exposure_rate END) AS pct_hackathon,
    MAX(CASE WHEN asset_name = 'hackathon' THEN exposure_rate_lift END) AS lift_hackathon,
    MAX(CASE WHEN asset_name = 'hackathon' THEN avg_exposure_value_exposed END) AS intensity_hackathon,
    MAX(CASE WHEN asset_name = 'devzone_download' THEN exposure_rate END) AS pct_devzone_download,
    MAX(CASE WHEN asset_name = 'devzone_download' THEN exposure_rate_lift END) AS lift_devzone_download,
    MAX(CASE WHEN asset_name = 'devzone_download' THEN avg_exposure_value_exposed END) AS intensity_devzone_download,
    MAX(CASE WHEN asset_name = 'ngc_download' THEN exposure_rate END) AS pct_ngc_download,
    MAX(CASE WHEN asset_name = 'ngc_download' THEN exposure_rate_lift END) AS lift_ngc_download,
    MAX(CASE WHEN asset_name = 'ngc_download' THEN avg_exposure_value_exposed END) AS intensity_ngc_download
FROM {CLUSTER_GROUP_DIFFERENTIATING_ASSET_TABLE}
GROUP BY 1
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
"""

cluster_group_composition_sql = f"""
CREATE OR REPLACE TABLE {CLUSTER_GROUP_COMPOSITION_TABLE} AS
WITH base AS (
    SELECT
        developer_id,
        {cluster_group_case} AS cluster_group,
        persona,
        behavior_journey_stage_30d,
        developer_effort_level
    FROM {CLUSTER_PROFILE_TABLE}
    WHERE {cluster_group_case} IS NOT NULL
),
group_sizes AS (
    SELECT cluster_group, COUNT(*) AS developers
    FROM base
    GROUP BY 1
),
persona_rank AS (
    SELECT cluster_group, persona AS persona_bucket, COUNT(*) AS developers,
           COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY cluster_group) AS share,
           ROW_NUMBER() OVER (PARTITION BY cluster_group ORDER BY COUNT(*) DESC, persona) AS rn
    FROM base
    GROUP BY 1,2
),
journey_rank AS (
    SELECT cluster_group, behavior_journey_stage_30d AS journey_bucket, COUNT(*) AS developers,
           COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY cluster_group) AS share,
           ROW_NUMBER() OVER (PARTITION BY cluster_group ORDER BY COUNT(*) DESC, behavior_journey_stage_30d) AS rn
    FROM base
    GROUP BY 1,2
),
effort_rank AS (
    SELECT cluster_group, developer_effort_level AS effort_bucket, COUNT(*) AS developers,
           COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY cluster_group) AS share,
           ROW_NUMBER() OVER (PARTITION BY cluster_group ORDER BY COUNT(*) DESC, developer_effort_level) AS rn
    FROM base
    GROUP BY 1,2
)
SELECT
    g.cluster_group,
    g.developers,
    MAX(CASE WHEN p.rn = 1 THEN p.persona_bucket END) AS top_persona_1,
    MAX(CASE WHEN p.rn = 1 THEN p.share END) AS top_persona_1_share,
    MAX(CASE WHEN p.rn = 2 THEN p.persona_bucket END) AS top_persona_2,
    MAX(CASE WHEN p.rn = 2 THEN p.share END) AS top_persona_2_share,
    MAX(CASE WHEN p.rn = 3 THEN p.persona_bucket END) AS top_persona_3,
    MAX(CASE WHEN p.rn = 3 THEN p.share END) AS top_persona_3_share,
    MAX(CASE WHEN j.rn = 1 THEN j.journey_bucket END) AS top_journey_1,
    MAX(CASE WHEN j.rn = 1 THEN j.share END) AS top_journey_1_share,
    MAX(CASE WHEN j.rn = 2 THEN j.journey_bucket END) AS top_journey_2,
    MAX(CASE WHEN j.rn = 2 THEN j.share END) AS top_journey_2_share,
    MAX(CASE WHEN e.rn = 1 THEN e.effort_bucket END) AS top_effort_1,
    MAX(CASE WHEN e.rn = 1 THEN e.share END) AS top_effort_1_share,
    MAX(CASE WHEN e.rn = 2 THEN e.effort_bucket END) AS top_effort_2,
    MAX(CASE WHEN e.rn = 2 THEN e.share END) AS top_effort_2_share
FROM group_sizes g
LEFT JOIN persona_rank p ON g.cluster_group = p.cluster_group
LEFT JOIN journey_rank j ON g.cluster_group = j.cluster_group
LEFT JOIN effort_rank e ON g.cluster_group = e.cluster_group
GROUP BY 1,2
ORDER BY CASE g.cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
"""

asset_audience_sql = f"""
CREATE OR REPLACE TABLE {ASSET_AUDIENCE_TABLE} AS
WITH asset_exposed AS (
    SELECT
        l.asset_name,
        l.asset_group,
        l.asset_role,
        COUNT(*) AS exposed_developers,
        AVG(CASE WHEN {cluster_group_case_b} = 'active' THEN 1.0 ELSE 0.0 END) AS pct_active,
        AVG(CASE WHEN {cluster_group_case_b} = 'cooling' THEN 1.0 ELSE 0.0 END) AS pct_cooling,
        AVG(CASE WHEN {cluster_group_case_b} = 'at_risk' THEN 1.0 ELSE 0.0 END) AS pct_at_risk,
        AVG(CASE WHEN b.behavior_journey_stage_30d = 'Builder' THEN 1.0 ELSE 0.0 END) AS pct_builder,
        AVG(CASE WHEN b.behavior_journey_stage_30d = 'Learner' THEN 1.0 ELSE 0.0 END) AS pct_learner,
        AVG(CASE WHEN b.developer_effort_level IN ('high effort', 'very high effort') THEN 1.0 ELSE 0.0 END) AS pct_high_effort,
        AVG(CASE WHEN b.persona = 'CUDA' THEN 1.0 ELSE 0.0 END) AS pct_cuda,
        AVG(CASE WHEN b.persona = 'GenAI' THEN 1.0 ELSE 0.0 END) AS pct_genai,
        AVG(CASE WHEN b.persona = 'Robotics' THEN 1.0 ELSE 0.0 END) AS pct_robotics,
        AVG(CASE WHEN b.persona = 'Simulation' THEN 1.0 ELSE 0.0 END) AS pct_simulation,
        AVG(CASE WHEN b.persona = 'Learning_Community' THEN 1.0 ELSE 0.0 END) AS pct_learning_community,
        AVG(CASE WHEN b.persona = 'Unknown' THEN 1.0 ELSE 0.0 END) AS pct_unknown_persona
    FROM {CLUSTER_ASSET_LONG_TABLE} l
    JOIN {CLUSTER_PROFILE_TABLE} b
      ON l.developer_id = b.developer_id
     AND l.cluster_id = b.cluster_id
     AND l.cluster_label = b.cluster_label
    WHERE l.analysis_role = 'rankable_activity_asset'
      AND l.exposed_flag = 1
    GROUP BY 1,2,3
)
SELECT
    asset_name,
    asset_group,
    asset_role,
    exposed_developers,
    pct_active,
    pct_cooling,
    pct_at_risk,
    pct_builder,
    pct_learner,
    pct_high_effort,
    CASE
        WHEN pct_cuda >= pct_genai AND pct_cuda >= pct_robotics AND pct_cuda >= pct_simulation AND pct_cuda >= pct_learning_community AND pct_cuda >= pct_unknown_persona THEN 'CUDA'
        WHEN pct_genai >= pct_robotics AND pct_genai >= pct_simulation AND pct_genai >= pct_learning_community AND pct_genai >= pct_unknown_persona THEN 'GenAI'
        WHEN pct_robotics >= pct_simulation AND pct_robotics >= pct_learning_community AND pct_robotics >= pct_unknown_persona THEN 'Robotics'
        WHEN pct_simulation >= pct_learning_community AND pct_simulation >= pct_unknown_persona THEN 'Simulation'
        WHEN pct_learning_community >= pct_unknown_persona THEN 'Learning_Community'
        ELSE 'Unknown'
    END AS top_persona
FROM asset_exposed
ORDER BY exposed_developers DESC, asset_name
"""

con.execute(cluster_group_asset_profile_sql)
con.execute(cluster_group_composition_sql)
con.execute(asset_audience_sql)

display(Markdown('### Lifecycle Group Asset Profile'))
display(con.execute(f"SELECT * FROM {CLUSTER_GROUP_ASSET_PROFILE_TABLE} ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END").fetchdf())

display(Markdown('### Lifecycle Group Composition Table'))
display(con.execute(f"SELECT * FROM {CLUSTER_GROUP_COMPOSITION_TABLE} ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END").fetchdf())

display(Markdown('### Asset Audience Table'))
display(con.execute(f"SELECT * FROM {ASSET_AUDIENCE_TABLE} ORDER BY exposed_developers DESC, asset_name").fetchdf())


In [33]:
con.close()